# Prueba de optimización del rendimiento — Danper

## Apartado 3.6 del Proyecto Integrador Big Data

**Objetivo:** comparar los tiempos de ejecución de una operación Spark sin caché y con caché.

**Dataset:** producción agrícola.

**Origen:** hdfs://localhost:9000/danper/silver/produccion

**Entorno:** Debian 12, PySpark 3.5.1, 1 CPU y 6.8 GiB de RAM.

**Métricas:** tiempo de ejecución en segundos y diferencia porcentual de rendimiento.

**Consideración:** las pruebas no modificarán los datos originales.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Danper_Optimizacion_Rendimiento")
    .master("local[1]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

ruta = "hdfs://localhost:9000/danper/silver/produccion"

df = spark.read.parquet(ruta)

print("===== CONFIGURACION =====")
print("Version Spark:", spark.version)
print("Dataset:", ruta)
print("Registros:", df.count())
print("Columnas:", len(df.columns))

print("\n===== ESQUEMA =====")
df.printSchema()

print("\n===== COLUMNAS DISPONIBLES =====")
print(df.columns)

print("\n===== PASO 103 FINALIZADO =====")

26/10/11 16:33:36 WARN Utils: Your hostname, Debian12 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 16:33:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 16:33:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 16:33:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

===== CONFIGURACION =====
Version Spark: 3.5.1
Dataset: hdfs://localhost:9000/danper/silver/produccion
Registros: 65922
Columnas: 13

===== ESQUEMA =====
root
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- codigo_cultivo: string (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- variable: string (nullable = true)
 |-- columna_origen: integer (nullable = true)
 |-- mes_origen: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- posicion_mensual: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes_numero: integer (nullable = true)
 |-- periodo: date (nullable = true)


===== COLUMNAS DISPONIBLES =====
['provincia', 'distrito', 'codigo_cultivo', 'cultivo', 'variable', 'columna_origen', 'mes_origen', 'valor', 'archivo_origen', 'posicion_mensual', 'anio', 'mes_numero', 'periodo']

===== PASO 103 FINALIZADO =====


## Paso 104. Primera medición: procesamiento sin caché

Se evalúa el tiempo de ejecución de una agregación mediante Apache Spark sin utilizar almacenamiento temporal en caché.

La operación agrupa los registros de producción agrícola por cultivo, año y variable, calculando la suma de los valores disponibles.

Se realizarán cinco ejecuciones para registrar los tiempos y calcular un promedio. Los resultados se utilizarán como línea base para comparar posteriormente el rendimiento con caché.

Las sumas son exclusivamente parte de una prueba técnica: no se interpretarán como indicadores de negocio porque las variables pueden tener unidades diferentes.

In [2]:
from pyspark.sql import functions as F
import time
import statistics

print("===== PASO 104: PRUEBA SIN CACHE =====")

# Nos aseguramos de comenzar sin caché
spark.catalog.clearCache()

# Misma operación que utilizaremos en ambas pruebas
def ejecutar_agregacion(datos):
    resultado = (
        datos
        .groupBy("cultivo", "anio", "variable")
        .agg(
            F.sum("valor").alias("valor_total"),
            F.count("*").alias("cantidad_registros")
        )
    )

    # collect() obliga a Spark a ejecutar la agregación completa
    # y devolver todos los grupos calculados.
    filas = resultado.collect()

    return len(filas)

tiempos_sin_cache = []
cantidad_grupos = None

for i in range(5):
    inicio = time.perf_counter()

    cantidad_grupos = ejecutar_agregacion(df)

    duracion = time.perf_counter() - inicio
    tiempos_sin_cache.append(duracion)

    print(f"Ejecución {i + 1}: {duracion:.4f} segundos")

promedio_sin_cache = statistics.mean(tiempos_sin_cache)
mediana_sin_cache = statistics.median(tiempos_sin_cache)

print("\n===== RESULTADOS SIN CACHE =====")
print("Registros de entrada:", 65922)
print("Grupos obtenidos:", cantidad_grupos)
print(f"Tiempo promedio: {promedio_sin_cache:.4f} segundos")
print(f"Tiempo mediano: {mediana_sin_cache:.4f} segundos")

print("\n===== PASO 104 FINALIZADO =====")

===== PASO 104: PRUEBA SIN CACHE =====


Ejecución 1: 4.4170 segundos
Ejecución 2: 0.9161 segundos
Ejecución 3: 0.8417 segundos
Ejecución 4: 0.8969 segundos
Ejecución 5: 0.6683 segundos

===== RESULTADOS SIN CACHE =====
Registros de entrada: 65922
Grupos obtenidos: 834
Tiempo promedio: 1.5480 segundos
Tiempo mediano: 0.8969 segundos

===== PASO 104 FINALIZADO =====


## Paso 105. Segunda medición: procesamiento con caché

Se aplica `cache()` sobre el DataFrame Silver de producción agrícola para evaluar si su reutilización mejora el rendimiento de una agregación repetida.

Primero se materializa la caché mediante `count()`. Posteriormente, se ejecuta cinco veces la misma agregación utilizada en el Paso 104.

Se registrarán el tiempo de preparación de la caché y los tiempos de las ejecuciones para distinguir el costo inicial del beneficio de reutilización.

In [3]:
from pyspark.sql import functions as F
import time
import statistics

print("===== PASO 105: PRUEBA CON CACHE =====")

# Crear un DataFrame destinado a la prueba con caché
df_cache = spark.read.parquet(ruta)

# Medir el costo de preparar y materializar la caché
inicio_cache = time.perf_counter()

df_cache.cache()
registros_cache = df_cache.count()

tiempo_preparacion_cache = time.perf_counter() - inicio_cache

print("Registros almacenados en caché:", registros_cache)
print(f"Tiempo de preparación: {tiempo_preparacion_cache:.4f} segundos")

# Reutilizar exactamente la misma operación del Paso 104
tiempos_con_cache = []
cantidad_grupos_cache = None

for i in range(5):
    inicio = time.perf_counter()

    cantidad_grupos_cache = ejecutar_agregacion(df_cache)

    duracion = time.perf_counter() - inicio
    tiempos_con_cache.append(duracion)

    print(f"Ejecución {i + 1}: {duracion:.4f} segundos")

promedio_con_cache = statistics.mean(tiempos_con_cache)
mediana_con_cache = statistics.median(tiempos_con_cache)

print("\n===== RESULTADOS CON CACHE =====")
print("Registros de entrada:", registros_cache)
print("Grupos obtenidos:", cantidad_grupos_cache)
print(f"Tiempo de preparación: {tiempo_preparacion_cache:.4f} segundos")
print(f"Tiempo promedio: {promedio_con_cache:.4f} segundos")
print(f"Tiempo mediano: {mediana_con_cache:.4f} segundos")

# Liberar la caché para no ocupar memoria innecesariamente
df_cache.unpersist(blocking=True)

print("\n===== CACHE LIBERADA =====")
print("===== PASO 105 FINALIZADO =====")

===== PASO 105: PRUEBA CON CACHE =====


Registros almacenados en caché: 65922
Tiempo de preparación: 3.0887 segundos
Ejecución 1: 0.9580 segundos
Ejecución 2: 0.5916 segundos
Ejecución 3: 0.4996 segundos
Ejecución 4: 0.5293 segundos
Ejecución 5: 0.4665 segundos

===== RESULTADOS CON CACHE =====
Registros de entrada: 65922
Grupos obtenidos: 834
Tiempo de preparación: 3.0887 segundos
Tiempo promedio: 0.6090 segundos
Tiempo mediano: 0.5293 segundos

===== CACHE LIBERADA =====
===== PASO 105 FINALIZADO =====


In [4]:
import statistics

print("===== PASO 106: COMPARACION FINAL =====")

# Comparación de las cinco ejecuciones
reduccion_promedio = (
    (promedio_sin_cache - promedio_con_cache)
    / promedio_sin_cache
) * 100

# Comparación sin la primera ejecución de cada prueba
promedio_sin_calentamiento = statistics.mean(
    tiempos_sin_cache[1:]
)

promedio_con_calentamiento = statistics.mean(
    tiempos_con_cache[1:]
)

reduccion_sin_calentamiento = (
    (promedio_sin_calentamiento - promedio_con_calentamiento)
    / promedio_sin_calentamiento
) * 100

# Tiempo total incluyendo la preparación de caché
total_sin_cache = sum(tiempos_sin_cache)

total_con_cache = (
    tiempo_preparacion_cache + sum(tiempos_con_cache)
)

print("\n===== PROMEDIO DE CINCO EJECUCIONES =====")
print(f"Sin caché: {promedio_sin_cache:.4f} s")
print(f"Con caché: {promedio_con_cache:.4f} s")
print(f"Reducción: {reduccion_promedio:.2f}%")

print("\n===== EXCLUYENDO PRIMERA EJECUCION =====")
print(f"Sin caché: {promedio_sin_calentamiento:.4f} s")
print(f"Con caché: {promedio_con_calentamiento:.4f} s")
print(f"Reducción: {reduccion_sin_calentamiento:.2f}%")

print("\n===== TIEMPOS TOTALES MEDIDOS =====")
print(f"Sin caché: {total_sin_cache:.4f} s")
print(f"Con caché (incluyendo preparación): {total_con_cache:.4f} s")

print("\n===== VALIDACION =====")
print(
    "Misma cantidad de grupos:",
    cantidad_grupos == cantidad_grupos_cache
)

print("\n===== PASO 106 FINALIZADO =====")

===== PASO 106: COMPARACION FINAL =====

===== PROMEDIO DE CINCO EJECUCIONES =====
Sin caché: 1.5480 s
Con caché: 0.6090 s
Reducción: 60.66%

===== EXCLUYENDO PRIMERA EJECUCION =====
Sin caché: 0.8307 s
Con caché: 0.5218 s
Reducción: 37.19%

===== TIEMPOS TOTALES MEDIDOS =====
Sin caché: 7.7400 s
Con caché (incluyendo preparación): 6.1337 s

===== VALIDACION =====
Misma cantidad de grupos: True

===== PASO 106 FINALIZADO =====


## Resultados y análisis de la optimización

### Configuración del experimento

Se utilizó Apache Spark 3.5.1 en un entorno on-premise con Debian 12, un procesador virtual y 6.8 GiB de memoria RAM.

La prueba se realizó sobre el conjunto de producción agrícola almacenado en la zona Silver de HDFS, en formato Parquet con compresión Snappy.

El conjunto contiene 65 922 registros y 13 columnas.

Se ejecutó una operación de agrupación por cultivo, año y variable, obteniendo 834 grupos. Se compararon cinco ejecuciones sin caché y cinco ejecuciones utilizando `cache()`.

### Resultados

| Indicador | Sin caché | Con caché |
|---|---:|---:|
| Tiempo promedio de cinco ejecuciones | 1.5480 s | 0.6090 s |
| Tiempo promedio sin la primera ejecución | 0.8307 s | 0.5218 s |
| Tiempo mediano | 0.8969 s | 0.5293 s |
| Tiempo total, incluyendo preparación de caché | 7.7400 s | 6.1337 s |

La reducción del tiempo promedio fue del 60.66 % considerando las cinco ejecuciones. Sin embargo, al excluir la primera ejecución de cada condición, la reducción fue del 37.19 %.

### Interpretación técnica

La utilización de caché permitió reutilizar temporalmente el DataFrame de producción agrícola, disminuyendo el tiempo de ejecución de las agregaciones repetidas.

La primera ejecución sin caché presentó un tiempo considerablemente mayor que las siguientes, posiblemente debido a la inicialización de tareas y al calentamiento del entorno Spark.

La preparación de la caché requirió 3.0887 segundos adicionales. Por lo tanto, su utilización debe evaluarse considerando la frecuencia de reutilización de los datos y la memoria disponible.

### Conclusión

El experimento mostró que el uso de caché puede mejorar el rendimiento de operaciones repetitivas sobre los datos Silver del proyecto Danper.

La reducción del 37.19 % en el promedio de las ejecuciones posteriores a la primera proporciona evidencia de una mejora en esta operación específica. No obstante, el resultado no demuestra una mejora equivalente del pipeline completo.

La conveniencia de esta optimización depende de la reutilización del conjunto de datos y de los recursos disponibles. En un entorno con memoria limitada, el almacenamiento temporal puede competir con otros procesos y reducir los beneficios esperados.